# Preparación de Textos para Prompt Engineering

**Objetivo**: Preparar dataset de evaluación con textos clínicos de próstata y sus entidades anotadas.

**Entregable**: `textos_evaluacion.json` con 20 textos para evaluar LLMs con prompt engineering.

---

## Configuración Inicial

In [1]:
import json
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd

In [2]:
# Rutas
BASE_PATH = Path('../datasets-taller/datasets-taller/prostata')
OUTPUT_PATH = Path('../resultados/fase6_prompt_engineering')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f"✓ Ruta del dataset: {BASE_PATH}")
print(f"✓ Ruta de salida: {OUTPUT_PATH}")

✓ Ruta del dataset: ..\datasets-taller\datasets-taller\prostata
✓ Ruta de salida: ..\resultados\fase6_prompt_engineering


## 1. Función para Leer Archivos BIO

In [3]:
def load_bio_file(file_path):
    """
    Lee archivo BIO y extrae oraciones con sus entidades.
    
    Returns:
        List[Dict]: Lista de oraciones con tokens, etiquetas y texto reconstruido
    """
    sentences = []
    current_tokens = []
    current_labels = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            
            if line == '':
                # Fin de oración
                if current_tokens:
                    sentences.append({
                        'tokens': current_tokens.copy(),
                        'labels': current_labels.copy(),
                        'texto': ' '.join(current_tokens)
                    })
                    current_tokens = []
                    current_labels = []
            else:
                # Línea con token y etiqueta
                parts = line.split('\t')
                if len(parts) == 2:
                    token, label = parts
                    current_tokens.append(token)
                    current_labels.append(label)
        
        # Última oración
        if current_tokens:
            sentences.append({
                'tokens': current_tokens,
                'labels': current_labels,
                'texto': ' '.join(current_tokens)
            })
    
    return sentences

In [4]:
def extract_entities_from_sentence(tokens, labels):
    """
    Extrae entidades nombradas de una oración en formato BIO.
    
    Returns:
        Dict: Diccionario con listas de entidades por tipo
    """
    entities = defaultdict(list)
    current_entity = []
    current_type = None
    
    for token, label in zip(tokens, labels):
        if label.startswith('B-'):
            # Guardar entidad anterior si existe
            if current_entity and current_type:
                entity_text = ' '.join(current_entity)
                entities[current_type].append(entity_text)
            
            # Iniciar nueva entidad
            current_type = label[2:]  # Quitar 'B-'
            current_entity = [token]
        
        elif label.startswith('I-'):
            # Continuar entidad actual
            if current_entity:
                current_entity.append(token)
        
        else:  # label == 'O'
            # Guardar entidad anterior si existe
            if current_entity and current_type:
                entity_text = ' '.join(current_entity)
                entities[current_type].append(entity_text)
                current_entity = []
                current_type = None
    
    # Guardar última entidad
    if current_entity and current_type:
        entity_text = ' '.join(current_entity)
        entities[current_type].append(entity_text)
    
    return dict(entities)

## 2. Cargar Dataset de Próstata

In [5]:
# Cargar archivos BIO
print("📂 Cargando archivos BIO...")
testing_sentences = load_bio_file(BASE_PATH / 'testing.bio')
validation_sentences = load_bio_file(BASE_PATH / 'validation.bio')

print(f"  - Testing: {len(testing_sentences)} oraciones")
print(f"  - Validation: {len(validation_sentences)} oraciones")

# Combinar ambos sets
all_sentences = testing_sentences + validation_sentences
print(f"\n✓ Total: {len(all_sentences)} oraciones disponibles")

📂 Cargando archivos BIO...
  - Testing: 991 oraciones
  - Validation: 929 oraciones

✓ Total: 1920 oraciones disponibles


In [6]:
# Ver ejemplo de oración
print("\n📝 Ejemplo de oración con entidades:\n")
ejemplo = all_sentences[5]
print(f"Texto: {ejemplo['texto'][:200]}...")
print(f"\nTokens: {ejemplo['tokens'][:15]}...")
print(f"Labels: {ejemplo['labels'][:15]}...")

entidades_ejemplo = extract_entities_from_sentence(ejemplo['tokens'], ejemplo['labels'])
print(f"\nEntidades extraídas:")
for tipo, ents in entidades_ejemplo.items():
    print(f"  - {tipo}: {ents}")


📝 Ejemplo de oración con entidades:

Texto: Tratamiento con radioterapia imrt radioterapia imrt a pelvis ....

Tokens: ['Tratamiento', 'con', 'radioterapia', 'imrt', 'radioterapia', 'imrt', 'a', 'pelvis', '.']...
Labels: ['O', 'O', 'B-TRATAMIENTO', 'I-TRATAMIENTO', 'B-TRATAMIENTO', 'I-TRATAMIENTO', 'O', 'O', 'O']...

Entidades extraídas:
  - TRATAMIENTO: ['radioterapia imrt', 'radioterapia imrt']


## 3. Seleccionar Textos Representativos

Criterios de selección:
- Oraciones con múltiples tipos de entidades
- Longitud entre 50-300 palabras
- Diversidad de entidades (biomarcadores, diagnósticos, tratamientos, etc.)

In [7]:
# Filtrar oraciones con entidades relevantes
textos_seleccionados = []

for idx, sent in enumerate(all_sentences):
    # Extraer entidades
    entities = extract_entities_from_sentence(sent['tokens'], sent['labels'])
    
    # Criterios de selección
    num_tokens = len(sent['tokens'])
    num_entity_types = len(entities)
    
    if 30 <= num_tokens <= 300 and num_entity_types >= 2:
        textos_seleccionados.append({
            'idx_original': idx,
            'texto': sent['texto'],
            'tokens': sent['tokens'],
            'labels': sent['labels'],
            'entidades': entities,
            'num_tokens': num_tokens,
            'num_entity_types': num_entity_types
        })

print(f"✓ Textos seleccionados con criterios: {len(textos_seleccionados)}")

# Ordenar por diversidad de entidades (más tipos primero)
textos_seleccionados.sort(key=lambda x: x['num_entity_types'], reverse=True)

# Tomar los mejores 15 textos
textos_top = textos_seleccionados[:15]
print(f"✓ Seleccionados top 15 textos más completos")

✓ Textos seleccionados con criterios: 278
✓ Seleccionados top 15 textos más completos


In [8]:
# Estadísticas de textos seleccionados
df_stats = pd.DataFrame([{
    'Texto ID': i+1,
    'Num Tokens': t['num_tokens'],
    'Tipos de Entidades': t['num_entity_types'],
    'Entidades': ', '.join(t['entidades'].keys())
} for i, t in enumerate(textos_top)])

print("\n📊 Estadísticas de textos seleccionados:\n")
print(df_stats.to_string(index=False))


📊 Estadísticas de textos seleccionados:

 Texto ID  Num Tokens  Tipos de Entidades                                                          Entidades
        1          61                   7 FECHA, CANCER, GLEASON, TNM, BIOMARCADOR, TRATAMIENTO, MEDICAMENTO
        2          68                   7 FECHA, CANCER, GLEASON, TNM, BIOMARCADOR, TRATAMIENTO, MEDICAMENTO
        3          51                   6                    CANCER, GLEASON, TNM, TRATAMIENTO, DOSIS, FECHA
        4          54                   6              CANCER, GLEASON, DOSIS, TNM, TRATAMIENTO, MEDICAMENTO
        5         126                   6               EDAD, CANCER, GLEASON, BIOMARCADOR, TNM, TRATAMIENTO
        6          58                   6                        CIRUGIA, FECHA, CANCER, GLEASON, DOSIS, TNM
        7          37                   6                     EDAD, CANCER, GLEASON, TNM, BIOMARCADOR, FECHA
        8          43                   6                     EDAD, CANCER, GLEASON, T

## 4. Agregar Texto de Ejemplo del Enunciado

In [9]:
# Texto de ejemplo proporcionado en el enunciado
texto_enunciado = """Paciente masculino de 72 años con antecedentes médicos de hipertensión arterial (HTA). 
Actualmente presenta un tumor prostático bilateral confirmado mediante biopsia. El diagnóstico histológico 
reveló un adenocarcinoma de próstata. El puntaje de Gleason reportado fue 3+3, lo que indica un cáncer de 
bajo grado. El PSA más alto registrado en la historia clínica fue de 9,9 ng/dL. La resonancia magnética no 
evidenció lesiones metastásicas ni afectación de la cápsula prostática. Las vesículas seminales se observaron 
normales en las imágenes por resonancia. No se detectaron adenopatías ni lesiones óseas sospechosas. La 
gammagrafía ósea fue negativa para metástasis. El cuadro clínico sugiere una neoplasia localizada de bajo riesgo."""

# Entidades esperadas (ground truth manual)
entidades_enunciado = {
    "EDAD": ["72 años"],
    "SEXO": ["masculino"],
    "ANTECEDENTES": ["hipertensión arterial", "HTA"],
    "CANCER": ["tumor prostático bilateral", "adenocarcinoma de próstata", "cáncer de bajo grado", "neoplasia localizada de bajo riesgo"],
    "GLEASON": ["Gleason 3+3"],
    "BIOMARCADOR": ["PSA 9,9 ng/dL"],
    "DIAGNOSTICO": ["diagnóstico histológico"],
    "CIRUGIA": ["biopsia"],
    "EXAMENES": ["resonancia magnética", "imágenes por resonancia", "gammagrafía ósea"],
    "HALLAZGOS": ["vesículas seminales normales", "no lesiones metastásicas", "no adenopatías", "no lesiones óseas", "negativa para metástasis"]
}

texto_ejemplo = {
    'texto_id': 0,
    'origen': 'enunciado',
    'texto': texto_enunciado.replace('\n', ' ').strip(),
    'entidades': entidades_enunciado
}

print("✓ Texto de ejemplo del enunciado agregado")
print(f"  Entidades: {list(entidades_enunciado.keys())}")

✓ Texto de ejemplo del enunciado agregado
  Entidades: ['EDAD', 'SEXO', 'ANTECEDENTES', 'CANCER', 'GLEASON', 'BIOMARCADOR', 'DIAGNOSTICO', 'CIRUGIA', 'EXAMENES', 'HALLAZGOS']


## 5. Preparar Dataset Final en Formato JSON

In [10]:
# Mapeo de etiquetas BIO a categorías del prompt
categoria_mapping = {
    'EDAD': 'edad',
    'BIOMARCADOR': 'biomarcadores',
    'CANCER': 'tipo_cancer',
    'GLEASON': 'biomarcadores',  # Gleason es un biomarcador
    'TNM': 'biomarcadores',
    'CIRUGIA': 'cirugias',
    'TRATAMIENTO': 'tratamiento',
    'MEDICAMENTO': 'medicacion',
    'DOSIS': 'medicacion',
    'FECHA': 'fecha'
}

def normalizar_entidades(entidades_bio):
    """Convierte entidades BIO al formato esperado por el prompt"""
    entidades_normalizadas = defaultdict(list)
    
    for tipo_bio, valores in entidades_bio.items():
        # Mapear a categoría del prompt
        categoria = categoria_mapping.get(tipo_bio, tipo_bio.lower())
        entidades_normalizadas[categoria].extend(valores)
    
    return dict(entidades_normalizadas)

In [11]:
# Crear dataset de evaluación
dataset_evaluacion = [texto_ejemplo]  # Empezar con el texto del enunciado

for i, texto_data in enumerate(textos_top, start=1):
    # Normalizar entidades
    entidades_norm = normalizar_entidades(texto_data['entidades'])
    
    dataset_evaluacion.append({
        'texto_id': i,
        'origen': 'dataset_prostata',
        'texto': texto_data['texto'],
        'entidades': entidades_norm,
        'metadata': {
            'num_tokens': texto_data['num_tokens'],
            'num_entity_types': texto_data['num_entity_types']
        }
    })

print(f"✓ Dataset de evaluación creado: {len(dataset_evaluacion)} textos")
print(f"  - 1 texto del enunciado")
print(f"  - {len(textos_top)} textos del dataset de próstata")

✓ Dataset de evaluación creado: 16 textos
  - 1 texto del enunciado
  - 15 textos del dataset de próstata


In [12]:
# Ver ejemplo del formato final
print("\n📄 Ejemplo del formato JSON final:\n")
print(json.dumps(dataset_evaluacion[0], indent=2, ensure_ascii=False)[:500] + "...")


📄 Ejemplo del formato JSON final:

{
  "texto_id": 0,
  "origen": "enunciado",
  "texto": "Paciente masculino de 72 años con antecedentes médicos de hipertensión arterial (HTA).  Actualmente presenta un tumor prostático bilateral confirmado mediante biopsia. El diagnóstico histológico  reveló un adenocarcinoma de próstata. El puntaje de Gleason reportado fue 3+3, lo que indica un cáncer de  bajo grado. El PSA más alto registrado en la historia clínica fue de 9,9 ng/dL. La resonancia magnética no  evidenció lesiones metastásicas n...


## 6. Guardar Dataset de Evaluación

In [13]:
# Guardar como JSON
output_file = OUTPUT_PATH / 'textos_evaluacion.json'

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(dataset_evaluacion, f, indent=2, ensure_ascii=False)

print(f"\n✅ Dataset guardado en: {output_file}")
print(f"   Tamaño del archivo: {output_file.stat().st_size / 1024:.2f} KB")
print(f"   Total de textos: {len(dataset_evaluacion)}")


✅ Dataset guardado en: ..\resultados\fase6_prompt_engineering\textos_evaluacion.json
   Tamaño del archivo: 14.11 KB
   Total de textos: 16


In [14]:
# Estadísticas finales
total_entidades = sum(len(t['entidades']) for t in dataset_evaluacion)
tipos_unicos = set()
for t in dataset_evaluacion:
    tipos_unicos.update(t['entidades'].keys())

print("\n📊 RESUMEN DEL DATASET DE EVALUACIÓN")
print("="*60)
print(f"Total de textos:        {len(dataset_evaluacion)}")
print(f"Total de entidades:     {total_entidades}")
print(f"Tipos únicos:           {len(tipos_unicos)}")
print(f"Categorías:             {', '.join(sorted(tipos_unicos))}")
print("="*60)
print("\n✅ Listo para usar en Kaggle!")
print("   Siguiente paso: Subir este archivo como dataset en Kaggle")
print("   Nombre sugerido: 'prostata-prompt-eval'")


📊 RESUMEN DEL DATASET DE EVALUACIÓN
Total de textos:        16
Total de entidades:     78
Tipos únicos:           17
Categorías:             ANTECEDENTES, BIOMARCADOR, CANCER, CIRUGIA, DIAGNOSTICO, EDAD, EXAMENES, GLEASON, HALLAZGOS, SEXO, biomarcadores, cirugias, edad, fecha, medicacion, tipo_cancer, tratamiento

✅ Listo para usar en Kaggle!
   Siguiente paso: Subir este archivo como dataset en Kaggle
   Nombre sugerido: 'prostata-prompt-eval'


## 7. Crear Estadísticas del Dataset

In [15]:
# Análisis por tipo de entidad
entidades_por_tipo = defaultdict(int)
for texto in dataset_evaluacion:
    for tipo, valores in texto['entidades'].items():
        entidades_por_tipo[tipo] += len(valores)

df_entidades = pd.DataFrame([
    {'Tipo de Entidad': tipo, 'Cantidad': cant}
    for tipo, cant in sorted(entidades_por_tipo.items(), key=lambda x: x[1], reverse=True)
])

print("\n📊 Distribución de entidades en el dataset:\n")
print(df_entidades.to_string(index=False))

# Guardar estadísticas
df_entidades.to_csv(OUTPUT_PATH / 'distribucion_entidades_eval.csv', index=False)
print(f"\n✓ Estadísticas guardadas en: {OUTPUT_PATH / 'distribucion_entidades_eval.csv'}")


📊 Distribución de entidades en el dataset:

Tipo de Entidad  Cantidad
  biomarcadores        45
          fecha        24
    tipo_cancer        23
     medicacion        15
    tratamiento        12
           edad         8
      HALLAZGOS         5
         CANCER         4
       EXAMENES         3
       cirugias         3
   ANTECEDENTES         2
           EDAD         1
           SEXO         1
        GLEASON         1
    BIOMARCADOR         1
    DIAGNOSTICO         1
        CIRUGIA         1

✓ Estadísticas guardadas en: ..\resultados\fase6_prompt_engineering\distribucion_entidades_eval.csv
